# Lab 3: Backpropagation, Vanishing Gradients, Weight Initialisation and Momentum

**DATA425 | Foundations of Deep Learning**

## What this lab is about

Lab 2 showed the training loop. This lab opens the part that often feels most mysterious: **where the gradients come from and why optimisation can become difficult in deep networks**.

We will work through:

- backpropagation as the chain rule applied carefully;
- why gradients can vanish or explode when many terms are multiplied together;
- how activation functions affect gradient flow;
- why initialisation matters;
- how momentum and Adam can make optimisation behave better.

The examples are small on purpose. Once the idea is clear in a tiny network, it becomes easier to understand what happens inside larger models.


## 0. Setup

Run the setup cells first. Most of the lab uses NumPy and Matplotlib. The final section uses Keras so we can connect the gradient ideas back to neural network training.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "tensorflow": "tensorflow",
    "sklearn": "scikit-learn"
}

missing = [package for module, package in REQUIRED.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Automatic installation failed. Install these packages manually or run this notebook "
            "in an environment with internet access: " + ", ".join(missing)
        ) from exc
else:
    print("All required packages are already installed.")

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
except RuntimeError:
    pass

SEED = 425
rng = np.random.default_rng(SEED)
tf.keras.utils.set_random_seed(SEED)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

The setup also fixes seeds and limits TensorFlow threading where possible. This keeps the notebook stable and avoids unnecessary runtime variation.


## 1. Backpropagation by hand

Consider a tiny network with one input value, one hidden value, and one loss:

$$a = wx + b,$$
$$h = \tanh(a),$$
$$L = (h - y)^2.$$

The forward pass computes $a$, then $h$, then the loss. The backward pass works in the opposite direction. It asks: **how did each earlier quantity contribute to the loss?**

This is exactly what backpropagation does in a large network, just with many more nodes and many more paths.


In [ ]:
x = 0.5
w = 2.0
b = -0.3
y = 0.4

a = w * x + b
h = np.tanh(a)
L = (h - y) ** 2

print(f"a = {a:.3f}")
print(f"h = tanh(a) = {h:.3f}")
print(f"loss = {L:.3f}")

These printed values are the forward pass. They tell us what the model predicted and how large the loss is before any update happens.


To update the weight, we need $\frac{dL}{dw}$. Instead of trying to jump straight from $L$ to $w$, we break the path into small pieces:

$$\frac{dL}{dw} = \frac{dL}{dh}\frac{dh}{da}\frac{da}{dw}.$$

This is the chain rule. Backpropagation is the chain rule organised so that each derivative is computed once and reused efficiently.

In words: the loss depends on $h$, $h$ depends on $a$, and $a$ depends on $w$. Multiplying these local sensitivities tells us how $w$ affects the loss.


In [ ]:
dL_dh = 2 * (h - y)
dh_da = 1 - np.tanh(a) ** 2
da_dw = x
da_db = 1.0

dL_dw = dL_dh * dh_da * da_dw
dL_db = dL_dh * dh_da * da_db

print(f"dL/dh = {dL_dh:.3f}")
print(f"dh/da = {dh_da:.3f}")
print(f"da/dw = {da_dw:.3f}")
print(f"dL/dw = {dL_dw:.3f}")
print(f"dL/db = {dL_db:.3f}")

learning_rate = 0.1
w_new = w - learning_rate * dL_dw
b_new = b - learning_rate * dL_db
print(f"Updated w = {w_new:.3f}")
print(f"Updated b = {b_new:.3f}")

The updated weight and bias move in the direction that reduces the loss for this example. A real network repeats this idea across many parameters and many mini-batches.


## 2. Why gradients can vanish or explode

In a deep network, backpropagation often multiplies many derivative terms together. This repeated multiplication can be fragile.

- If many terms are smaller than 1, the product can shrink toward 0. This is a vanishing gradient.
- If many terms are larger than 1, the product can grow very large. This is an exploding gradient.

Vanishing gradients make early layers learn very slowly. Exploding gradients can make training unstable because updates become too large.


In [ ]:
layers_count = np.arange(1, 81)
small_factor = 0.5 ** layers_count
medium_factor = 0.9 ** layers_count
large_factor = 1.1 ** layers_count

plt.semilogy(layers_count, small_factor, label="multiply by 0.5 each layer")
plt.semilogy(layers_count, medium_factor, label="multiply by 0.9 each layer")
plt.semilogy(layers_count, large_factor, label="multiply by 1.1 each layer")
plt.xlabel("number of repeated factors")
plt.ylabel("product size, log scale")
plt.title("Repeated multiplication can shrink or grow gradients")
plt.legend()
plt.show()

The log-scale plot shows how quickly repeated multiplication changes a signal. Even a factor like 0.9 can become tiny when it is multiplied many times.


### Activation derivatives

Activation functions do more than shape the forward pass. Their derivatives also shape the backward pass.

Sigmoid derivatives are never larger than 0.25, and they are much smaller when the input is far from 0. This means sigmoid units can saturate, which causes weak gradients.

Tanh has a larger maximum derivative, but it can also saturate. ReLU has a simple derivative: 1 for positive inputs and 0 for negative inputs. This is one reason ReLU became popular in deep networks.


In [ ]:
z = np.linspace(-6, 6, 500)
sigmoid = 1 / (1 + np.exp(-z))
sigmoid_derivative = sigmoid * (1 - sigmoid)
tanh_derivative = 1 - np.tanh(z) ** 2
relu_derivative = (z > 0).astype(float)

plt.plot(z, sigmoid_derivative, label="sigmoid derivative")
plt.plot(z, tanh_derivative, label="tanh derivative")
plt.plot(z, relu_derivative, label="ReLU derivative")
plt.xlabel("pre-activation value")
plt.ylabel("derivative")
plt.title("Activation derivatives affect gradient flow")
plt.legend()
plt.show()

Notice that sigmoid and tanh have very small derivatives in the flat parts of their curves. If many layers sit in those flat regions, learning can slow down dramatically.


## 3. Weight initialisation

Initial weights control the scale of signals as data moves forward through the network and gradients move backward through it.

If the weights are too small, activations can shrink layer by layer. If the weights are too large, activations can blow up or saturate. Good initialisation tries to keep signal scale roughly stable across layers.

Two common ideas are:

- **Xavier-style initialisation**, often paired with tanh-like activations;
- **He-style initialisation**, often paired with ReLU-like activations.

The exact formulas are less important than the principle: choose an initial scale that matches the layer width and activation function.


In [ ]:
def activation_std_through_layers(scale, activation="tanh", width=128, depth=40, seed=SEED):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(512, width))
    stds = []
    for _ in range(depth):
        W = rng.normal(0, scale, size=(width, width))
        X = X @ W
        if activation == "tanh":
            X = np.tanh(X)
        elif activation == "relu":
            X = np.maximum(X, 0)
        stds.append(float(X.std()))
    return np.array(stds)

depth = 40
std_tiny = activation_std_through_layers(scale=0.01, activation="tanh", depth=depth)
std_large = activation_std_through_layers(scale=1.00, activation="tanh", depth=depth)
std_xavier = activation_std_through_layers(scale=np.sqrt(1 / 128), activation="tanh", depth=depth)
std_he = activation_std_through_layers(scale=np.sqrt(2 / 128), activation="relu", depth=depth)

plt.plot(std_tiny, label="tiny weights")
plt.plot(std_large, label="large weights")
plt.plot(std_xavier, label="Xavier-style tanh")
plt.plot(std_he, label="He-style ReLU")
plt.xlabel("layer")
plt.ylabel("activation standard deviation")
plt.title("Initialisation changes signal flow")
plt.legend()
plt.show()

The plot shows that initialisation changes how information flows through many layers. Good initialisation does not guarantee a good model, but bad initialisation can make training much harder.


## 4. Momentum and Adam on a simple loss surface

Plain gradient descent only uses the current gradient. Momentum keeps a moving average of past gradients, which can help the optimiser continue in a useful direction instead of zigzagging.

Adam combines two ideas:

- momentum, which smooths the direction of movement;
- adaptive scaling, which changes step sizes based on recent gradient magnitudes.

The next example uses a narrow bowl-shaped loss surface. This shape is useful because plain gradient descent can bounce across the steep direction while moving slowly along the flat direction.


In [ ]:
def bowl_loss(point):
    x, y = point
    return 0.02 * x**2 + y**2

def bowl_grad(point):
    x, y = point
    return np.array([0.04 * x, 2.0 * y])

def optimise(method="sgd", steps=80, lr=0.2):
    p = np.array([8.0, 5.0], dtype=float)
    path = [p.copy()]
    v = np.zeros_like(p)
    m = np.zeros_like(p)
    s = np.zeros_like(p)
    beta1, beta2 = 0.9, 0.999
    for t in range(1, steps + 1):
        g = bowl_grad(p)
        if method == "sgd":
            update = g
        elif method == "momentum":
            v = 0.9 * v + g
            update = v
        elif method == "adam":
            m = beta1 * m + (1 - beta1) * g
            s = beta2 * s + (1 - beta2) * (g ** 2)
            m_hat = m / (1 - beta1 ** t)
            s_hat = s / (1 - beta2 ** t)
            update = m_hat / (np.sqrt(s_hat) + 1e-8)
        p = p - lr * update
        path.append(p.copy())
    return np.array(path)

paths = {
    "SGD": optimise("sgd", lr=0.4),
    "Momentum": optimise("momentum", lr=0.08),
    "Adam": optimise("adam", lr=0.25),
}

xs = np.linspace(-9, 9, 250)
ys = np.linspace(-6, 6, 250)
xx, yy = np.meshgrid(xs, ys)
zz = 0.02 * xx**2 + yy**2

plt.contour(xx, yy, zz, levels=30)
for name, path in paths.items():
    plt.plot(path[:, 0], path[:, 1], marker="o", markersize=2, label=name)
plt.xlabel("w1")
plt.ylabel("w2")
plt.title("Optimiser paths on a narrow bowl")
plt.legend()
plt.show()

Compare the paths rather than just the final point. Optimisers can behave differently even when they are trying to minimise the same loss.


## 5. Keras comparison: sigmoid versus ReLU

We finish by training two deep networks on the same moons dataset. The only difference is the hidden activation function.

This is not a universal rule that ReLU always wins. It is a demonstration of an important practical point: activation functions affect optimisation, not just model expressiveness.

A deep sigmoid network can have weaker gradients, especially if many hidden units saturate. ReLU often gives stronger gradient flow in hidden layers, which is why it is a common default.


In [ ]:
X, y = make_moons(n_samples=600, noise=0.20, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

def build_deep_classifier(activation):
    model = keras.Sequential([keras.Input(shape=(2,))])
    for _ in range(5):
        model.add(layers.Dense(16, activation=activation))
    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model

histories = {}
for activation in ["sigmoid", "relu"]:
    tf.keras.utils.set_random_seed(SEED)
    model = build_deep_classifier(activation)
    history = model.fit(X_train_s, y_train, validation_data=(X_test_s, y_test),
                        epochs=15, batch_size=32, verbose=0)
    histories[activation] = history.history
    print(f"{activation} final validation accuracy: {history.history['val_accuracy'][-1]:.3f}")

plt.plot(histories["sigmoid"]["val_loss"], label="sigmoid validation loss")
plt.plot(histories["relu"]["val_loss"], label="ReLU validation loss")
plt.xlabel("epoch")
plt.ylabel("validation loss")
plt.title("Activation choice affects optimisation")
plt.legend()
plt.show()

Look at the validation loss curves. If one model learns faster or reaches a lower loss, connect that behaviour back to the derivative plots and the vanishing gradient idea.


## Summary

In this lab, you saw that:

- Backpropagation is the chain rule applied from the loss back through the network.
- Deep networks multiply many derivative terms, so gradients can vanish or explode.
- Activation functions affect gradient flow through their derivatives.
- Weight initialisation helps keep activations and gradients at useful scales.
- Momentum and Adam can improve optimisation by using information from past gradients.

The main takeaway is that training a neural network is not only about choosing a powerful architecture. It is also about making sure useful gradient information can move through that architecture.
